In [ ]:
import os
import pyvista as pv
import numpy as np
from scipy.spatial.transform import Rotation as R

# 1. Force VTK to use EGL/Surfaceless rendering for headless environments
os.environ["VTK_DEFAULT_OPENGL_WINDOW"] = "vtkEGLRenderWindow"
os.environ["EGL_PLATFORM"] = "surfaceless"

# 2. Set backend to client-side HTML rendering
pv.set_jupyter_backend('html')

def plot_trajectory_static_html():
    t = np.linspace(0, 4 * np.pi, 200)
    positions = np.column_stack([np.cos(t), np.sin(t), t * 0.2])
    orientations = np.column_stack([np.sin(t) * 0.2, np.cos(t) * 0.2, t])

    plotter = pv.Plotter(notebook=True)
    plotter.add_axes()
    plotter.show_grid(xtitle='X (m)', ytitle='Y (m)', ztitle='Z (m)')

    # Add trajectory line
    traj_line = pv.PolyData(positions)
    plotter.add_mesh(traj_line, color='yellow', line_width=3, label='Trajectory')

    # Add head sphere
    head_sphere = pv.Sphere(radius=0.08, center=positions[-1])
    plotter.add_mesh(head_sphere, color='white')

    # Add orientation arrows along the path
    axis_length = 0.3
    axis_colors = ['red', 'green', 'blue']
    local_dirs = np.eye(3)

    for i in range(0, len(positions), 10): # every 10 steps
        pos = positions[i]
        rot = R.from_euler('xyz', orientations[i]).as_matrix()
        world_dirs = rot @ local_dirs

        for c_idx in range(3):
            arrow = pv.Arrow(start=pos, direction=world_dirs[:, c_idx], scale=axis_length)
            plotter.add_mesh(arrow, color=axis_colors[c_idx])

    plotter.show()

plot_trajectory_static_html()

In [2]:
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from scipy.spatial.transform import Rotation as R

# Force Plotly to embed inside an iframe wrapper for JupyterLab/Headless compatibility
pio.renderers.default = "iframe"

def build_3d_animated_trajectory(positions, orientations, axis_length=0.3, step=2):
    n_frames = len(positions)
    local_dirs = np.eye(3) * axis_length
    colors = ['red', 'green', 'blue']

    rot0 = R.from_euler('xyz', orientations[0]).as_matrix()
    dirs0 = rot0 @ local_dirs

    path_trace = go.Scatter3d(
        x=positions[:1, 0], y=positions[:1, 1], z=positions[:1, 2],
        mode='lines', line=dict(color='yellow', width=5), name='Path'
    )

    head_trace = go.Scatter3d(
        x=[positions[0, 0]], y=[positions[0, 1]], z=[positions[0, 2]],
        mode='markers', marker=dict(size=6, color='white'), name='Head'
    )

    arrow_traces = []
    for c_idx in range(3):
        arrow_traces.append(go.Scatter3d(
            x=[positions[0, 0], positions[0, 0] + dirs0[0, c_idx]],
            y=[positions[0, 1], positions[0, 1] + dirs0[1, c_idx]],
            z=[positions[0, 2], positions[0, 2] + dirs0[2, c_idx]],
            mode='lines', line=dict(color=colors[c_idx], width=6),
            name=['X (Forward)', 'Y (Left)', 'Z (Up)'][c_idx]
        ))

    frames = []
    for k in range(0, n_frames, step):
        pos = positions[k]
        rot = R.from_euler('xyz', orientations[k]).as_matrix()
        dirs = rot @ local_dirs

        frame_data = [
            go.Scatter3d(x=positions[:k+1, 0], y=positions[:k+1, 1], z=positions[:k+1, 2]),
            go.Scatter3d(x=[pos[0]], y=[pos[1]], z=[pos[2]]),
        ]

        for c_idx in range(3):
            frame_data.append(go.Scatter3d(
                x=[pos[0], pos[0] + dirs[0, c_idx]],
                y=[pos[1], pos[1] + dirs[1, c_idx]],
                z=[pos[2], pos[2] + dirs[2, c_idx]]
            ))

        frames.append(go.Frame(data=frame_data, name=f'frame_{k}'))

    fig = go.Figure(
        data=[path_trace, head_trace] + arrow_traces,
        layout=go.Layout(
            scene=dict(
                xaxis_title='X (m)', yaxis_title='Y (m)', zaxis_title='Z (m)',
                aspectmode='data'
            ),
            updatemenus=[{
                'type': 'buttons',
                'showactive': False,
                'y': 0, 'x': 0.1, 'xanchor': 'right', 'yanchor': 'top',
                'buttons': [
                    {'label': '▶ Play', 'method': 'animate', 'args': [None, {'frame': {'duration': 30, 'redraw': True}, 'fromcurrent': True}]},
                    {'label': '❚❚ Pause', 'method': 'animate', 'args': [[None], {'frame': {'duration': 0, 'redraw': False}, 'mode': 'immediate'}]}
                ]
            }],
            sliders=[{
                'steps': [{'args': [[f.name], {'frame': {'duration': 0, 'redraw': True}, 'mode': 'immediate'}],
                           'label': str(idx * step), 'method': 'animate'} for idx, f in enumerate(frames)],
                'x': 0.1, 'len': 0.9, 'y': 0
            }]
        ),
        frames=frames
    )

    return fig

# --- Run ---
t = np.linspace(0, 4 * np.pi, 150)
positions = np.column_stack([np.cos(t), np.sin(t), t * 0.2])
orientations = np.column_stack([np.sin(t) * 0.2, np.cos(t) * 0.2, t])

fig = build_3d_animated_trajectory(positions, orientations, axis_length=0.3, step=2)
fig.show()